# D1-S2-Nb4: Simple Liquid Handling
Showcase:
- state-verification:
    - plate presence
- auto-correction / self-healing aPs
    - plate offset

## Set your `protocol_mode`

...so it is easy to switch between simulation and execution

In [1]:
prep_simulation = True  # True or False

star_simulation = True  # True or False

flex_simulation = False  # True or False

num_samples = 96  # how many raw samples come in: 1 to 192 (2 plates)

run_identifier = "nb4"

## Library Import

In [2]:
import asyncio
import logging
import time

---
## Visualizer3D
...so everything that follows has one highest-level reference frame

In [3]:
from pylabrobot.resources import Coordinate, Resource
from pylabrobot.visualizer3D.facility import Facility
from pylabrobot.visualizer3D.server import Viewer3D

facility = Facility(name="facility", size_x=2_400, size_y=1_000, size_z=0)

viewer = Viewer3D(root = facility, name="nb4_normalisation")

await viewer.start()

viewer on http://127.0.0.1:1338/#token=SnBoMrcmhHU-6XQSqjTjsd9UUG1K_yLbCUSxNSFQv4M  (websocket 2122)


2026-09-27 18:15:05,295 - pylabrobot.visualizer3D.server - WARNING - refused a websocket without this viewer's token: a page left open from an earlier viewer, or one opened without the link it printed. Repeats are logged at debug.


viewer: a browser connected, drawing with WebGPU


---

## Set your `logging`
We (and our agents) always benefit from a log.

This follows standard Python logging but for convenience we set up:
- showing logging inside notebook using `verbose` declaration
- simple way to log to a file (in append mode)

I recommend always logging all commands you send/receive from any device for auditing purposes.

In [ ]:
import datetime
import logging
import time

import pylabrobot
from pylabrobot.io import LOG_LEVEL_IO

# one mode per device: 8 combinations, each with its own log folder
prep_mode = "simulation" if prep_simulation else "execution"
star_mode = "simulation" if star_simulation else "execution"
flex_mode = "simulation" if flex_simulation else "execution"
protocol_mode = f"prep_{prep_mode}-star_{star_mode}-flex_{flex_mode}"

started_at = datetime.datetime.now().strftime("%Y%m%d-%H%M%S")
log_path = f"_logs/{protocol_mode}/{run_identifier}_{started_at}.log"
pylabrobot.setup_logger(log_path, level=LOG_LEVEL_IO)

pylabrobot.verbose(True, level=logging.INFO)

print(f"logging to {log_path}")
logging.getLogger("pylabrobot").info("--- %s (%s) ---", run_identifier, protocol_mode)

## Device setups
### (shadow) STAR(let)

In [4]:
from pylabrobot.hamilton.star import STARlet

star = STARlet(simulation=star_simulation)

facility.assign_child_resource(
    resource=star,
    location=Coordinate(x=800.0, y=0.0, z=0.0)
)

await star.setup()

2026-09-27 18:15:07,348 - pylabrobot.hamilton.star.driver.master - INFO - [Hamilton STAR] Connected on simulation (no link)
  Firmware: master 7.6S 25 2021_11_05 (GRU C0), pipettes 4.0S j 2022-03-16, x_arm 1.4S 2012-04-25, head96 5.0S i 2021-10-22 (H0 XE167), iswap 4.1S 2011-12-19, autoload 3.4S f 2017-01-09
  Arms: 1
  Configuration: 30 slots
  Autoload: 1D barcode scanner
    left: hamilton_legacy_star_dual_rail_arm, 354.0 mm wide, travel 95.0 to 800.2 mm, workspace -323.2 to 977.2 mm
      channels: 8 (1000uL) | 96-head: 96 head II | 384-head: none | iSWAP: wide gripper


### Opentrons Flex
The Flex has no simulator: its deck is built either way, and the robot is only reached when `flex_simulation = False`.
Then `setup()` creates a run and **homes all axes**.

In [5]:
from pylabrobot.opentrons import Flex
from pylabrobot.resources.opentrons import FlexDeck

flex_deck = FlexDeck()

facility.assign_child_resource(
    resource=flex_deck,
    location=Coordinate(x=-900.0, y=0.0, z=0.0)
)

flex = None
if not flex_simulation:
    flex = Flex(serial_port="/dev/tty.usbmodem1201", deck=flex_deck)  # Flex connected over USB serial
    await flex.setup()

RuntimeError: No Flex found on the local network.

### Prep

In [ ]:
from pylabrobot.hamilton.prep import Prep
from pylabrobot.hamilton.prep.driver.simulator import RECORDING_PREP_HEAD8

prep = Prep(
    simulation=prep_simulation,
    declared_configuration_json=RECORDING_PREP_HEAD8,  # our Prep, with its 8-channel head
    )

facility.assign_child_resource(
    prep,
    location=Coordinate(x=0.0, y=0.0, z=0.0)
)

await prep.setup()

---
## Deck setup

we are now starting to build more complex setups and need more resources:

In [ ]:
from pylabrobot.resources import set_tip_tracking, set_volume_tracking

set_tip_tracking(enabled=True)
set_volume_tracking(enabled=True)

In [ ]:
import math

from pylabrobot.resources import ContainerRack, ResourceHolder, create_ordered_items_2d
from pylabrobot.resources.biorad import biorad_96_wellplate_200uL_Vb
from pylabrobot.resources.corning import cor_axy_1_troughplate_rowVb
from pylabrobot.resources.hamilton import (
    Trough_CAR_5R60_A00,
    hamilton_1_trough_60mL_Vb,
    hamilton_96_tiprack_50uL_NTR,
    hamilton_96_tiprack_300uL_NTR,
    hamilton_plate_carrier_L5_ac,
    hamilton_tip_carrier_L5_ntr_a00,
)

In [ ]:
def hamilton_5_troughrack_60mL(name: str) -> ContainerRack:
    """5x 60 mL troughs, evenly spaced and centred on an SBS footprint."""
    n_troughs, trough_x, trough_y = 5, 19.0, 90.0
    gap_x = (127.76 - n_troughs * trough_x) / (n_troughs + 1)
    return ContainerRack(
        name=name,
        size_x=127.76,
        size_y=85.48,
        size_z=40.0,
        ordered_items=create_ordered_items_2d(
            klass=ResourceHolder,
            num_items_x=n_troughs,
            num_items_y=1,
            dx=gap_x,
            dy=-5.52,  # a 90 mm trough overhangs the 85.48 mm footprint to the front
            dz=0.0,  # TODO: measure where the trough's base sits in the rack
            item_dx=trough_x + gap_x,
            item_dy=trough_y,
            size_x=trough_x,
            size_y=trough_y,
            size_z=0.0,
            name_prefix=name,
        ),
    )

### `agonist`: one name on every device
Same name everywhere, prefixed with the device: `prep_agonist`, `star_agonist`, `flex_agonist`.

In [ ]:
# Prep: the 60 mL trough in position 0 of the trough rack
prep_troughrack = prep.deck[0] = hamilton_5_troughrack_60mL(name="prep_troughrack")
prep_troughrack[0] = prep_agonist = hamilton_1_trough_60mL_Vb(name="prep_agonist")

# STAR: a 60 mL trough carrier, the 60 mL trough in position 0
star_trough_carrier = Trough_CAR_5R60_A00(name="star_trough_carrier")
star_trough_carrier[0] = star_agonist = hamilton_1_trough_60mL_Vb(name="star_agonist")
star.deck.assign_child_resource(star_trough_carrier, track=20)

# Flex: the 290 mL single-well reservoir
flex_agonist = cor_axy_1_troughplate_rowVb(name="flex_agonist")
flex_deck.assign_child_at_slot(flex_agonist, "D1")

# what is in each reservoir at the start, in uL
prep_agonist.tracker.set_volume(40_000)
star_agonist.tracker.set_volume(40_000)
flex_agonist.get_well("A1").tracker.set_volume(100_000)

### Sample plates
Each device has 2 dedicated positions for sample plates: in this simple example the plate the samples come in is also the plate they leave in.
How many plates are made follows from `num_samples`, declared at the top.

In [ ]:
n_plates = math.ceil(num_samples / 96)
assert 1 <= n_plates <= 2, f"{num_samples} samples need {n_plates} plates; each device has 2 plate positions"


def new_plates(device: str) -> list:
    """Sample plates as they arrive: 100 uL in every well."""
    plates = [biorad_96_wellplate_200uL_Vb(name=f"{device}_sample_plate_{i}") for i in range(n_plates)]
    for plate in plates:
        plate.set_well_volumes([100.0] * plate.num_items)
    return plates

In [ ]:
# Prep: deck sites 4, 5, right of the agonist's trough rack (site 0)
prep_plate_sites = [4, 5]

prep_sample_plates = new_plates("prep")
for site, plate in zip(prep_plate_sites, prep_sample_plates):
    prep.deck[site] = plate

In [ ]:
# STAR: a plate carrier right of the agonist's trough carrier (track 20); positions 0, 1
star_plate_sites = [0, 1]

star_plate_carrier = hamilton_plate_carrier_L5_ac(name="star_plate_carrier")
star.deck.assign_child_resource(star_plate_carrier, track=21)

star_sample_plates = new_plates("star")
for site, plate in zip(star_plate_sites, star_sample_plates):
    star_plate_carrier[site] = plate

In [ ]:
# Flex: slots D2, C2, right of the agonist (D1)
flex_plate_slots = ["D2", "C2"]

flex_sample_plates = new_plates("flex")
for slot, plate in zip(flex_plate_slots, flex_sample_plates):
    flex_deck.assign_child_at_slot(plate, slot)

then create new tipracks and assign them to the deck:

In [ ]:
prep_tips_50uL_00 = prep.deck[1] = hamilton_96_tiprack_50uL_NTR(name="prep_tips_50uL_00")

prep_tips_300uL_00 = prep.deck[7] = hamilton_96_tiprack_300uL_NTR(name="prep_tips_300uL_00")

same on the STAR:

In [ ]:
star_tip_carrier = hamilton_tip_carrier_L5_ntr_a00(name="star_tip_carrier")
star_tip_carrier[0] = star_tips_50uL_00 = hamilton_96_tiprack_50uL_NTR(name="star_tips_50uL_00")
star.deck.assign_child_resource(star_tip_carrier, track=13)

and on the Flex:

In [ ]:
from pylabrobot.resources.opentrons import flex_96_tiprack_50ul

flex_tips_50uL_00 = flex_96_tiprack_50ul(name="flex_tips_50uL_00")
flex_deck.assign_child_at_slot(flex_tips_50uL_00, "C1")

---
## A very simple serial dilution
Every sample well starts with 100 uL. Each multichannel (the Prep's 8-channel head, the STAR's 8 channels, the Flex's P50 8-channel on the right):
1. picks up a column of 50 uL tips,
2. aspirates 50 uL of agonist,
3. for each column, 1 to 12:
    - dispenses the 50 uL into it,
    - mixes: 5 aspirate/dispense cycles of 50 uL, 4 mm above the well bottom, no LLD,
    - aspirates 50 uL to carry into the next column (not after column 12),
4. returns the tips.

N.B.: the 50 uL tips' water liquid classes all blow out, so the Hamilton calls ask for `blow_out`.

In [ ]:
# STAR: its 8 channels
channels = [0, 1, 2, 3, 4, 5, 6, 7]
tips = star_tips_50uL_00.column(0)
plate = star_sample_plates[0]
no_lld = star.pipettes.LLDMode.OFF

# the liquid class these calls look up for themselves; pass one to choose another
# from pylabrobot.hamilton.star.liquid_classes import get_star_liquid_class
# from pylabrobot.resources.liquid import Liquid
# water_50uL = get_star_liquid_class(
#     tip_volume=50, is_core=False, is_tip=True, has_filter=False,
#     liquid=Liquid.WATER, jet=False, blow_out=True,
# )

await star.pipettes.pick_up_tips(tip_spots=tips, use_channels=channels)

# agonist into column 1
await star.pipettes.aspirate(
    containers=[star_agonist] * 8,
    volumes=[50] * 8,
    use_channels=channels,
    lld_mode=no_lld,
    blow_out=[True] * 8,
    # hamilton_liquid_classes=[water_50uL] * 8,
)

# column to column: dispense, mix, aspirate
for col in range(12):
    column = plate.column(col)
    await star.pipettes.dispense(
        containers=column,
        volumes=[50] * 8,
        use_channels=channels,
        liquid_heights=[4] * 8,
        blow_out=[True] * 8,
        # hamilton_liquid_classes=[water_50uL] * 8,
    )

    # mix: 5 cycles, 4 mm above the well bottom
    for cycle in range(5):
        await star.pipettes.aspirate(
            containers=column,
            volumes=[50] * 8,
            use_channels=channels,
            liquid_heights=[4] * 8,
            lld_mode=no_lld,
            blow_out=[True] * 8,
        )
        await star.pipettes.dispense(
            containers=column,
            volumes=[50] * 8,
            use_channels=channels,
            liquid_heights=[4] * 8,
            blow_out=[True] * 8,
        )

    # carry 50 uL into the next column
    if col < 11:
        await star.pipettes.aspirate(
            containers=column,
            volumes=[50] * 8,
            use_channels=channels,
            liquid_heights=[4] * 8,
            lld_mode=no_lld,
            blow_out=[True] * 8,
        )

# the tips go back where they came from
await star.pipettes.return_tips(use_channels=channels)

In [ ]:
async def prep_dilution():
    # Prep: its 8-channel head
    tips = prep_tips_50uL_00.column(0)
    plate = prep_sample_plates[0]
    no_lld = prep.pipettes.LLDMode.OFF

    # the liquid class these calls look up for themselves; pass one to choose another
    # from pylabrobot.hamilton.star.liquid_classes import get_star_liquid_class
    # from pylabrobot.resources.liquid import Liquid
    # water_50uL = get_star_liquid_class(
    #     tip_volume=50, is_core=False, is_tip=True, has_filter=False,
    #     liquid=Liquid.WATER, jet=False, blow_out=True,
    # )

    await prep.head8.pick_up_tips(tip_spots=tips)

    # agonist into column 1
    await prep.head8.aspirate(
        containers=[prep_agonist],
        volume=50,
        lld_mode=no_lld,
        blow_out=True,
        # hamilton_liquid_classes=water_50uL,
    )

    # column to column: dispense, mix, aspirate
    for col in range(12):

        column = plate.column(col)

        await prep.head8.dispense(
            containers=column,
            volume=50,
            liquid_height=4,
            blow_out=True,
            # hamilton_liquid_classes=water_50uL,
        )

        # mix: 5 cycles, 4 mm above the well bottom
        for cycle in range(5):
            await prep.head8.aspirate(
                containers=column, volume=50, liquid_height=4, lld_mode=no_lld, blow_out=True
            )
            await prep.head8.dispense(containers=column, volume=50, liquid_height=4, blow_out=True)

        # carry 50 uL into the next column
        if col < 11:
            await prep.head8.aspirate(
                containers=column, volume=50, liquid_height=4, lld_mode=no_lld, blow_out=True
            )

    # the tips go back where they came from
    await prep.head8.drop_tips(destinations=tips)

In [ ]:
async def flex_dilution():
    # Flex: the P50 8-channel on the right mount (only with a real Flex: it has no simulator)
    if flex is None:
        return
    p50 = flex.right
    plate = flex_sample_plates[0]

    await p50.pick_up_tips(target=flex_tips_50uL_00.column(0))

    # agonist into column 1
    await p50.aspirate_container(container=flex_agonist.get_well("A1"), volume=50)

    # column to column: dispense, mix, aspirate
    for col in range(12):
        column = plate.column(col)
        await p50.dispense(target=column, volume=50, liquid_height=4)

        # mix: 5 cycles, 4 mm above the well bottom
        for cycle in range(5):
            await p50.aspirate(target=column, volume=50, liquid_height=4)
            await p50.dispense(target=column, volume=50, liquid_height=4)

        # carry 50 uL into the next column
        if col < 11:
            await p50.aspirate(target=column, volume=50, liquid_height=4)

    # the tips go back where they came from
    await p50.drop_tips(target=flex_tips_50uL_00, column=1)

In [ ]:
# the Prep and the Flex run at the same time, each on its own
await asyncio.gather(prep_dilution(), flex_dilution())

---
# Shutdown

In [ ]:
await star.stop()

In [ ]:
await prep.stop()

In [ ]:
if flex is not None:
    await flex.stop()  # homes and ends the run: it moves the robot

In [ ]:
await viewer.stop()